# GĐ2 — Pipeline thật 2 stage trên Kaggle T4 ×2: TiMePReSt vs PipeDream (VGG-16 / CIFAR-100)
Yêu cầu: Accelerator = **GPU T4 x2**, Internet = **On**, đã *Add Input* Kaggle Dataset chứa thư mục `cifar-100-python/`.
Chạy lần lượt; dán output cell 2, 3 và bảng cuối cell 5 cho agent.

## 1. Repo + cài đặt + môi trường

In [ ]:
import os
REPO = '/kaggle/working/timeprest-reproduction'
if not os.path.exists(REPO):
    !git clone https://github.com/cotda/timeprest-reproduction.git {REPO}
%cd {REPO}
!git pull --ff-only
!git log --oneline -1
!pip install -q -e ".[test]"
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
!python -c "import torch; print('torch', torch.__version__, '| cuda', torch.version.cuda, '| gpus', torch.cuda.device_count(), '| nccl', torch.cuda.nccl.version())"
!find /kaggle/input -maxdepth 4 -name cifar-100-python

## 2. Unit test (CPU, gồm test 2 tiến trình) ~1–2 phút

In [ ]:
!python -m pytest -q

## 3. Check GĐ2 trên 2 GPU (D1–D4) ~5 phút

In [ ]:
!torchrun --standalone --nproc_per_node=2 -m timeprest.dist.checks --config configs/kaggle_quick.yaml

## 4. Chạy dài 160 epoch (TiMePReSt rồi PipeDream), checkpoint trong /kaggle/working/runs
Nếu session bị ngắt: chạy lại cell 1 rồi đúng cell này (có `--resume`).

In [ ]:
!torchrun --standalone --nproc_per_node=2 -m timeprest.dist.train --config configs/kaggle_timeprest.yaml --resume

In [ ]:
!torchrun --standalone --nproc_per_node=2 -m timeprest.dist.train --config configs/kaggle_pipedream.yaml --resume

## 5. So sánh + đóng gói kết quả để tải về

In [ ]:
RUNS = '/kaggle/working/runs'
!python -m timeprest.plot --runs {RUNS}/kaggle_cifar100_vgg16_timeprest {RUNS}/kaggle_cifar100_vgg16_pipedream --out {RUNS}/compare_kaggle --title "VGG-16 / CIFAR-100, real 2-GPU pipeline (Kaggle T4x2)"
!cd {RUNS} && zip -r /kaggle/working/results_phase2.zip */metrics.csv */summary.json */config.json */op_trace_epoch1.json */checks.json compare_kaggle*
from IPython.display import Image, FileLink
display(Image(f'{RUNS}/compare_kaggle.png')); display(FileLink('/kaggle/working/results_phase2.zip'))

### Nếu NCCL vẫn bị treo
`configs/kaggle_base.yaml` đã đặt sẵn `NCCL_P2P_DISABLE=1`. Nếu vẫn treo, chạy chẩn đoán (sau 120 s mỗi worker in stack mọi thread, dừng hẳn ở 150 s) rồi dán output:
`!TIMEPREST_STACK_DUMP_S=120 NCCL_DEBUG=INFO NCCL_DEBUG_SUBSYS=INIT,P2P timeout 150 torchrun --standalone --nproc_per_node=2 -m timeprest.dist.checks --config configs/kaggle_quick.yaml --only 1 2>&1 | tail -200`